# Large state readability fixture

This notebook is intentionally broad. It has many cells, mixed markdown and
code, long sources, stdout with structured sections, one HTML table display,
and a late stale path.

The task for the state reader is simple: identify the cell ids, statuses, and
the stale path quickly without needing to scan all source.


In [1]:
import json
import math
import random
import statistics
import sys
from collections import Counter, defaultdict

random.seed(20260615)
print("setup complete")
print("seed=20260615")


setup complete
seed=20260615


## Field guide

The synthetic rows below mimic a real agent notebook where each cell carries a
specific observation. The notebook has enough cells that an agent should rely on
the compact state outline, not source scanning.

Important labels:

- `control`: baseline records.
- `candidate`: records that might need deeper inspection.
- `watch`: records that look acceptable but are close to a cutoff.


In [2]:
rows = []
for i in range(84):
    segment = ["control", "candidate", "watch", "candidate"][i % 4]
    risk = round(0.18 + (i % 13) * 0.041 + (0.07 if segment == "candidate" else 0), 3)
    rows.append({
        "id": f"evt-{i:03d}",
        "segment": segment,
        "risk": min(risk, 0.94),
        "latency_ms": 90 + (i * 17) % 230,
        "tokens": 220 + (i * 29) % 900,
    })

print(f"rows={len(rows)}")
print("segment counts", dict(Counter(row["segment"] for row in rows)))
print("first", rows[0])
print("last", rows[-1])


rows=84
segment counts {'control': 21, 'candidate': 42, 'watch': 21}
first {'id': 'evt-000', 'segment': 'control', 'risk': 0.18, 'latency_ms': 90, 'tokens': 220}
last {'id': 'evt-083', 'segment': 'candidate', 'risk': 0.455, 'latency_ms': 121, 'tokens': 827}


In [3]:
print("stream: validator phases")
for phase in ["parse", "schema", "sample", "aggregate", "budget", "final"]:
    print(f"phase={phase:<10} status=ok elapsed_ms={len(phase) * 37}")
print("stream: representative rows")
for row in rows[:12]:
    print(f"{row['id']} | {row['segment']:<9} | risk={row['risk']:.3f} | latency={row['latency_ms']}")
print("stream: warnings")
for idx in range(1, 9):
    print(f"warning[{idx:02d}] synthetic long output line used to test stdout summary readability")


stream: validator phases
phase=parse      status=ok elapsed_ms=185
phase=schema     status=ok elapsed_ms=222
phase=sample     status=ok elapsed_ms=222
phase=aggregate  status=ok elapsed_ms=333
phase=budget     status=ok elapsed_ms=222
phase=final      status=ok elapsed_ms=185
stream: representative rows
evt-000 | control   | risk=0.180 | latency=90
evt-001 | candidate | risk=0.291 | latency=107
evt-002 | watch     | risk=0.262 | latency=124
evt-003 | candidate | risk=0.373 | latency=141
evt-004 | control   | risk=0.344 | latency=158
evt-005 | candidate | risk=0.455 | latency=175
evt-006 | watch     | risk=0.426 | latency=192
evt-007 | candidate | risk=0.537 | latency=209
evt-008 | control   | risk=0.508 | latency=226
evt-009 | candidate | risk=0.619 | latency=243
evt-010 | watch     | risk=0.590 | latency=260
evt-011 | candidate | risk=0.701 | latency=277
stream: warnings
warning[01] synthetic long output line used to test stdout summary readability
warning[02] synthetic long output li

## Early observation memo

The early rows are balanced enough to proceed. This note is intentionally short
so it should be easy to skim past in `state --outputs none`.


In [4]:
feature_catalog = [
    ('feature_00', 0.110, 'segment_0', 'derived from rolling window 0'),
    ('feature_01', 0.127, 'segment_1', 'derived from rolling window 1'),
    ('feature_02', 0.144, 'segment_2', 'derived from rolling window 2'),
    ('feature_03', 0.161, 'segment_3', 'derived from rolling window 3'),
    ('feature_04', 0.178, 'segment_4', 'derived from rolling window 4'),
    ('feature_05', 0.195, 'segment_0', 'derived from rolling window 5'),
    ('feature_06', 0.212, 'segment_1', 'derived from rolling window 6'),
    ('feature_07', 0.229, 'segment_2', 'derived from rolling window 0'),
    ('feature_08', 0.246, 'segment_3', 'derived from rolling window 1'),
    ('feature_09', 0.263, 'segment_4', 'derived from rolling window 2'),
    ('feature_10', 0.280, 'segment_0', 'derived from rolling window 3'),
    ('feature_11', 0.297, 'segment_1', 'derived from rolling window 4'),
    ('feature_12', 0.314, 'segment_2', 'derived from rolling window 5'),
    ('feature_13', 0.331, 'segment_3', 'derived from rolling window 6'),
    ('feature_14', 0.348, 'segment_4', 'derived from rolling window 0'),
    ('feature_15', 0.365, 'segment_0', 'derived from rolling window 1'),
    ('feature_16', 0.382, 'segment_1', 'derived from rolling window 2'),
    ('feature_17', 0.399, 'segment_2', 'derived from rolling window 3'),
    ('feature_18', 0.416, 'segment_3', 'derived from rolling window 4'),
    ('feature_19', 0.433, 'segment_4', 'derived from rolling window 5'),
    ('feature_20', 0.450, 'segment_0', 'derived from rolling window 6'),
    ('feature_21', 0.467, 'segment_1', 'derived from rolling window 0'),
    ('feature_22', 0.484, 'segment_2', 'derived from rolling window 1'),
    ('feature_23', 0.501, 'segment_3', 'derived from rolling window 2'),
    ('feature_24', 0.518, 'segment_4', 'derived from rolling window 3'),
    ('feature_25', 0.535, 'segment_0', 'derived from rolling window 4'),
    ('feature_26', 0.552, 'segment_1', 'derived from rolling window 5'),
    ('feature_27', 0.569, 'segment_2', 'derived from rolling window 6'),
    ('feature_28', 0.586, 'segment_3', 'derived from rolling window 0'),
    ('feature_29', 0.603, 'segment_4', 'derived from rolling window 1'),
    ('feature_30', 0.620, 'segment_0', 'derived from rolling window 2'),
    ('feature_31', 0.637, 'segment_1', 'derived from rolling window 3'),
    ('feature_32', 0.654, 'segment_2', 'derived from rolling window 4'),
    ('feature_33', 0.671, 'segment_3', 'derived from rolling window 5'),
    ('feature_34', 0.688, 'segment_4', 'derived from rolling window 6'),
    ('feature_35', 0.705, 'segment_0', 'derived from rolling window 0'),
    ('feature_36', 0.722, 'segment_1', 'derived from rolling window 1'),
    ('feature_37', 0.739, 'segment_2', 'derived from rolling window 2'),
    ('feature_38', 0.756, 'segment_3', 'derived from rolling window 3'),
    ('feature_39', 0.773, 'segment_4', 'derived from rolling window 4'),
    ('feature_40', 0.790, 'segment_0', 'derived from rolling window 5'),
    ('feature_41', 0.807, 'segment_1', 'derived from rolling window 6'),
    ('feature_42', 0.824, 'segment_2', 'derived from rolling window 0'),
    ('feature_43', 0.841, 'segment_3', 'derived from rolling window 1'),
    ('feature_44', 0.858, 'segment_4', 'derived from rolling window 2'),
    ('feature_45', 0.875, 'segment_0', 'derived from rolling window 3'),
    ('feature_46', 0.892, 'segment_1', 'derived from rolling window 4'),
    ('feature_47', 0.909, 'segment_2', 'derived from rolling window 5'),
    ('feature_48', 0.926, 'segment_3', 'derived from rolling window 6'),
    ('feature_49', 0.943, 'segment_4', 'derived from rolling window 0'),
    ('feature_50', 0.960, 'segment_0', 'derived from rolling window 1'),
    ('feature_51', 0.977, 'segment_1', 'derived from rolling window 2'),
    ('feature_52', 0.994, 'segment_2', 'derived from rolling window 3'),
    ('feature_53', 1.011, 'segment_3', 'derived from rolling window 4'),
    ('feature_54', 1.028, 'segment_4', 'derived from rolling window 5'),
    ('feature_55', 1.045, 'segment_0', 'derived from rolling window 6'),
    ('feature_56', 1.062, 'segment_1', 'derived from rolling window 0'),
    ('feature_57', 1.079, 'segment_2', 'derived from rolling window 1'),
    ('feature_58', 1.096, 'segment_3', 'derived from rolling window 2'),
    ('feature_59', 1.113, 'segment_4', 'derived from rolling window 3'),
    ('feature_60', 1.130, 'segment_0', 'derived from rolling window 4'),
    ('feature_61', 1.147, 'segment_1', 'derived from rolling window 5'),
    ('feature_62', 1.164, 'segment_2', 'derived from rolling window 6'),
    ('feature_63', 1.181, 'segment_3', 'derived from rolling window 0'),
    ('feature_64', 1.198, 'segment_4', 'derived from rolling window 1'),
    ('feature_65', 1.215, 'segment_0', 'derived from rolling window 2'),
    ('feature_66', 1.232, 'segment_1', 'derived from rolling window 3'),
    ('feature_67', 1.249, 'segment_2', 'derived from rolling window 4'),
    ('feature_68', 1.266, 'segment_3', 'derived from rolling window 5'),
    ('feature_69', 1.283, 'segment_4', 'derived from rolling window 6'),
    ('feature_70', 1.300, 'segment_0', 'derived from rolling window 0'),
    ('feature_71', 1.317, 'segment_1', 'derived from rolling window 1'),
]

weighted_total = sum(weight for _, weight, _, _ in feature_catalog)
by_segment = defaultdict(float)
for name, weight, segment, note in feature_catalog:
    by_segment[segment] += weight

print("feature catalog entries", len(feature_catalog))
print("weighted_total", round(weighted_total, 3))
for segment, total in sorted(by_segment.items()):
    print(f"feature_weight segment={segment} total={total:.3f}")

top_features = sorted(feature_catalog, key=lambda item: item[1], reverse=True)[:5]
print("top features")
for item in top_features:
    print(item)


feature catalog entries 72
weighted_total 51.372
feature_weight segment=segment_0 total=10.575
feature_weight segment=segment_1 total=10.830
feature_weight segment=segment_2 total=9.751
feature_weight segment=segment_3 total=9.989
feature_weight segment=segment_4 total=10.227
top features
('feature_71', 1.317, 'segment_1', 'derived from rolling window 1')
('feature_70', 1.3, 'segment_0', 'derived from rolling window 0')
('feature_69', 1.283, 'segment_4', 'derived from rolling window 6')
('feature_68', 1.266, 'segment_3', 'derived from rolling window 5')
('feature_67', 1.249, 'segment_2', 'derived from rolling window 4')


In [5]:
from IPython.display import HTML, display

segment_summary = []
for segment in sorted({row["segment"] for row in rows}):
    values = [row["risk"] for row in rows if row["segment"] == segment]
    segment_summary.append((segment, len(values), round(statistics.mean(values), 3), round(max(values), 3)))

html_rows = "\n".join(
    f"<tr><td>{segment}</td><td>{count}</td><td>{mean}</td><td>{max_risk}</td></tr>"
    for segment, count, mean, max_risk in segment_summary
)
display(HTML(f"<table><tr><th>segment</th><th>n</th><th>mean risk</th><th>max risk</th></tr>{html_rows}</table>"))
print("html table displayed with segment risk summary")


segment,n,mean risk,max risk
candidate,42,0.487,0.742
control,21,0.424,0.672
watch,21,0.405,0.672


html table displayed with segment risk summary


## Metric interpretation

The HTML table should render as a short preview in `state --outputs summary`.
The full source is not useful for the agent once the cell description and output
preview are visible.


In [6]:
sample = rows[10:20]
print("sample rows")
for row in sample:
    payload = json.dumps(row, sort_keys=True)
    print(payload)


sample rows
{"id": "evt-010", "latency_ms": 260, "risk": 0.59, "segment": "watch", "tokens": 510}
{"id": "evt-011", "latency_ms": 277, "risk": 0.701, "segment": "candidate", "tokens": 539}
{"id": "evt-012", "latency_ms": 294, "risk": 0.672, "segment": "control", "tokens": 568}
{"id": "evt-013", "latency_ms": 311, "risk": 0.25, "segment": "candidate", "tokens": 597}
{"id": "evt-014", "latency_ms": 98, "risk": 0.221, "segment": "watch", "tokens": 626}
{"id": "evt-015", "latency_ms": 115, "risk": 0.332, "segment": "candidate", "tokens": 655}
{"id": "evt-016", "latency_ms": 132, "risk": 0.303, "segment": "control", "tokens": 684}
{"id": "evt-017", "latency_ms": 149, "risk": 0.414, "segment": "candidate", "tokens": 713}
{"id": "evt-018", "latency_ms": 166, "risk": 0.385, "segment": "watch", "tokens": 742}
{"id": "evt-019", "latency_ms": 183, "risk": 0.496, "segment": "candidate", "tokens": 771}


## Checkpoint notes

Nothing is stale yet. Later cells will deliberately become stale after a
threshold update.


In [7]:
rollups = {}
for segment in sorted({row["segment"] for row in rows}):
    segment_rows = [row for row in rows if row["segment"] == segment]
    rollups[segment] = {
        "n": len(segment_rows),
        "risk_mean": round(statistics.mean(row["risk"] for row in segment_rows), 3),
        "token_mean": round(statistics.mean(row["tokens"] for row in segment_rows), 1),
        "latency_p90": sorted(row["latency_ms"] for row in segment_rows)[int(len(segment_rows) * 0.9) - 1],
    }

for segment, values in rollups.items():
    print(segment, values)


candidate {'n': 42, 'risk_mean': 0.487, 'token_mean': 645.1, 'latency_p90': 293}
control {'n': 21, 'risk_mean': 0.424, 'token_mean': 608.6, 'latency_p90': 284}
watch {'n': 21, 'risk_mean': 0.405, 'token_mean': 666.6, 'latency_p90': 276}


In [8]:
audit_events = []
for i, row in enumerate(rows[::7]):
    event = {
        "event": "audit_sample",
        "ordinal": i,
        "id": row["id"],
        "segment": row["segment"],
        "risk_band": "high" if row["risk"] >= 0.62 else "normal",
    }
    audit_events.append(event)
    print(json.dumps(event, sort_keys=True))
print("audit event count", len(audit_events))


{"event": "audit_sample", "id": "evt-000", "ordinal": 0, "risk_band": "normal", "segment": "control"}
{"event": "audit_sample", "id": "evt-007", "ordinal": 1, "risk_band": "normal", "segment": "candidate"}
{"event": "audit_sample", "id": "evt-014", "ordinal": 2, "risk_band": "normal", "segment": "watch"}
{"event": "audit_sample", "id": "evt-021", "ordinal": 3, "risk_band": "normal", "segment": "candidate"}
{"event": "audit_sample", "id": "evt-028", "ordinal": 4, "risk_band": "normal", "segment": "control"}
{"event": "audit_sample", "id": "evt-035", "ordinal": 5, "risk_band": "normal", "segment": "candidate"}
{"event": "audit_sample", "id": "evt-042", "ordinal": 6, "risk_band": "normal", "segment": "watch"}
{"event": "audit_sample", "id": "evt-049", "ordinal": 7, "risk_band": "high", "segment": "candidate"}
{"event": "audit_sample", "id": "evt-056", "ordinal": 8, "risk_band": "normal", "segment": "control"}
{"event": "audit_sample", "id": "evt-063", "ordinal": 9, "risk_band": "high", "s

## Long narrative review

This markdown cell is intentionally longer than a normal agent note. Its role is
to test whether `state --outputs none` still lets the reader jump by cell id and
status without absorbing all prose.

- Checkpoint 01: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 02: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 03: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 04: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 05: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 06: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 07: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 08: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 09: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 10: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 11: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 12: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 13: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 14: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 15: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 16: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 17: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 18: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 19: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 20: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 21: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 22: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 23: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 24: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 25: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 26: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 27: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 28: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 29: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 30: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 31: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 32: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 33: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 34: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 35: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 36: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 37: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 38: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 39: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 40: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 41: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 42: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 43: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 44: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.
- Checkpoint 45: this note intentionally repeats the same finding with a slightly different slice name so the state renderer has to budget source text.


In [9]:
print("stdout before warning")
for i in range(5):
    print(f"stdout checkpoint {i}: records={len(rows)}")
print("stderr warning: synthetic warning routed to stderr", file=sys.stderr)
print("stderr detail: no action required", file=sys.stderr)
print("stdout after warning")


stdout before warning
stdout checkpoint 0: records=84
stdout checkpoint 1: records=84
stdout checkpoint 2: records=84
stdout checkpoint 3: records=84
stdout checkpoint 4: records=84
stdout after warning


stderr warning: synthetic warning routed to stderr
stderr detail: no action required


In [10]:
candidates = []
for row in rows:
    score = round(0.55 * row["risk"] + 0.0009 * row["tokens"] + (0.04 if row["segment"] == "candidate" else 0), 3)
    if row["segment"] in {"candidate", "watch"}:
        candidates.append({"id": row["id"], "segment": row["segment"], "score": score})

candidates = sorted(candidates, key=lambda item: item["score"], reverse=True)[:14]
print("candidate score table")
for item in candidates:
    print(f"{item['id']} {item['segment']:<9} score={item['score']:.3f}")


candidate score table
evt-061 candidate score=1.361
evt-062 watch     score=1.331
evt-025 candidate score=1.299
evt-031 candidate score=1.297
evt-059 candidate score=1.263
evt-023 candidate score=1.201
evt-029 candidate score=1.200
evt-030 watch     score=1.170
evt-051 candidate score=1.167
evt-057 candidate score=1.166
evt-058 watch     score=1.136
evt-021 candidate score=1.104
evt-027 candidate score=1.103
evt-022 watch     score=1.074


## Pre-decision note

The candidate table is stable. The next code cell records the operating cutoff.


In [14]:
cutoff = 0.72
flagged = [item for item in candidates if item["score"] >= cutoff]
print("cutoff", cutoff)
print("flagged_count", len(flagged))
for item in flagged:
    print(f"flagged {item['id']} score={item['score']:.3f}")


cutoff 0.72
flagged_count 14
flagged evt-061 score=1.361
flagged evt-062 score=1.331
flagged evt-025 score=1.299
flagged evt-031 score=1.297
flagged evt-059 score=1.263
flagged evt-023 score=1.201
flagged evt-029 score=1.200
flagged evt-030 score=1.170
flagged evt-051 score=1.167
flagged evt-057 score=1.166
flagged evt-058 score=1.136
flagged evt-021 score=1.104
flagged evt-027 score=1.103
flagged evt-022 score=1.074


## Threshold finding

Using cutoff `0.64`, the current flagged set is accepted for follow-up review.
This markdown should become stale when the cutoff cell is updated.


In [12]:
review_queue = [item["id"] for item in flagged]
print("review queue")
for item_id in review_queue:
    print(item_id)
print("queue size", len(review_queue))


review queue
evt-061
evt-062
evt-025
evt-031
evt-059
evt-023
evt-029
evt-030
evt-051
evt-057
evt-058
evt-021
evt-027
evt-022
queue size 14


In [13]:
print("final index")
for idx, label in enumerate(["setup", "rows", "features", "scores", "threshold", "queue"]):
    print(f"{idx}: {label}")
print("final queue size", len(review_queue))


final index
0: setup
1: rows
2: features
3: scores
4: threshold
5: queue
final queue size 14
